In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/LeonelSalgueiro/PROJETO_BURNOUT/refs/heads/main/machine_learning/student_mental_health_burnout_1M.csv"

# O pandas vai ler as vírgulas e criar as colunas automaticamente
df = pd.read_csv(url)

# Verifique se o número de colunas está correto (deve dar algo como 20)
print(f"Colunas detectadas: {df.shape[1]}")

# Isso deve mostrar uma tabela bonitinha, não uma linha de texto
df.head()

Colunas detectadas: 20


,age,gender,academic_year,study_hours_per_day,exam_pressure,academic_performance,stress_level,anxiety_score,depression_score,sleep_hours,physical_activity,social_support,screen_time,internet_usage,financial_stress,family_expectation,burnout_score,mental_health_index,risk_level,dropout_risk
0,23,Male,2,5.596071,6.487218,68.411114,4.116950,2.275713,1.986730,6.880545,2.728861,6.470080,4.993801,4.983157,3.446626,3.586147,2.037344,7.074487,Low,1.746601
1,20,Male,3,5.597171,5.631481,67.682159,0.349489,0.000000,0.000000,7.463339,3.690866,0.000000,3.862980,5.136124,2.814039,5.478666,0.000000,9.860204,Low,0.000000
2,29,Male,2,2.580491,6.015297,58.372363,3.476177,2.425201,0.851996,8.946670,3.296720,6.901725,5.428880,3.058333,4.918515,6.068155,0.000000,7.626370,Low,0.696941
3,27,Male,4,4.607208,6.684005,68.925653,6.778843,4.512425,4.285645,4.571380,2.065480,2.349857,6.304842,6.931147,6.915885,6.557540,7.227651,4.649042,High,5.380592
4,24,Male,4,2.186569,4.010945,69.141915,1.854595,1.102558,0.000000,5.989324,4.026504,4.512921,4.903146,5.134903,4.382820,5.934779,0.000000,8.927394,Low,0.000000


In [3]:
# Reduzindo o dataset para 10.000 linhas aleatórias para agilizar o MVP
df = df.sample(n=10000, random_state=7)
print(f"Novo tamanho do dataset: {df.shape}")

Novo tamanho do dataset: (10000, 20)


In [4]:
# Definindo as colunas que vamos usar (X) e a que queremos prever (y)
# Note que tirei 'burnout_score' e 'dropout_risk' para o modelo não "viciar"
features = ['age', 'gender', 'academic_year', 'study_hours_per_day', 'exam_pressure',
            'academic_performance', 'stress_level', 'anxiety_score', 'depression_score',
            'sleep_hours', 'physical_activity', 'social_support', 'screen_time',
            'internet_usage', 'financial_stress', 'family_expectation']

X = df[features].copy()
y = df['risk_level']

In [5]:
from sklearn.preprocessing import LabelEncoder

# Transformando o gênero em 0 e 1
le = LabelEncoder()
X['gender'] = le.fit_transform(X['gender'])

# Transformando o risk_level em número.
y = le.fit_transform(y)

In [6]:
from sklearn.model_selection import train_test_split

# Separando 20% para teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=7)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Criando a lista de modelos para comparar
models = []
models.append(('KNN', KNeighborsClassifier()))
models.append(('CART', DecisionTreeClassifier()))
models.append(('NB', GaussianNB()))
models.append(('SVM', SVC(cache_size=700, max_iter=10000)))

# Loop para treinar e avaliar cada um usando Cross-Validation (exigência do roteiro)
results = []
names = []

for name, model in models:
    # Criando um pipeline que primeiro padroniza (StandardScaler) e depois treina
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    cv_results = cross_val_score(pipe, X_train, y_train, cv=10) # 10-fold cross-validation
    results.append(cv_results)
    names.append(name)
    print(f"{name}: {cv_results.mean():.4f} ({cv_results.std():.4f})")

KNN: 0.8313 (0.0170)
CART: 0.8027 (0.0148)
NB: 0.8389 (0.0141)
SVM: 0.8590 (0.0130)


In [8]:
from sklearn.model_selection import GridSearchCV

# Definindo os parâmetros para testar no SVM
# C: controla o erro (valores menores suavizam a margem)
# kernel: a função matemática que separa os dados
param_grid = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['rbf', 'linear']
}

# Criamos o pipeline específico para o SVM
pipe_svm = Pipeline([('scaler', StandardScaler()), ('model', SVC())])

# Configurando a busca (GridSearch)
grid = GridSearchCV(estimator=pipe_svm, param_grid=param_grid, scoring='accuracy', cv=5)
grid.fit(X_train, y_train)

# Resultados da Otimização
print(f"Melhor acurácia após GridSearch: {grid.best_score_:.4f}")
print(f"Melhores parâmetros: {grid.best_params_}")

# Salvando o modelo campeão
melhor_modelo = grid.best_estimator_

Melhor acurácia após GridSearch: 0.8658
Melhores parâmetros: {'model__C': 1, 'model__kernel': 'linear'}
